# DDPM training notebook

Train a small 2D epsilon network, save weights to
`src/lessons/ddpm/assets/ddpm-weights.json`, and produce four validation
plots:
- A: 1000 reverse-sample endpoints, chi-squared on 4-cluster uniformity
- B: learned score field at t = {1, 25, 50, 75, 99}
- C: 10 x_hat_0 trajectories
- D: histogram of sample-to-nearest-center distance

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import json
import matplotlib.pyplot as plt
from pathlib import Path

torch.manual_seed(0)
np.random.seed(0)

# Data: 4-cluster 2D mixture
centers = np.array([[2.0, 2.0], [2.0, -2.0], [-2.0, 2.0], [-2.0, -2.0]])
N_per = 250
X = np.vstack([np.random.normal(c, 0.2, (N_per, 2)) for c in centers]).astype(np.float32)
Xt = torch.tensor(X)

# Noise schedule (T = 100 for browser speed)
T = 100
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

# Architecture: 4-layer MLP, hidden 64, time embedding 32
HIDDEN = 64
TIME_DIM = 32

class EpsNet(nn.Module):
    def __init__(self, hidden=HIDDEN, time_dim=TIME_DIM):
        super().__init__()
        self.time_dim = time_dim
        self.net = nn.Sequential(
            nn.Linear(2 + time_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2),
        )
    def time_embed(self, t):
        half = self.time_dim // 2
        freqs = torch.exp(-np.log(10000) * torch.arange(half).float() / (half - 1))
        emb = t[:, None].float() * freqs[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)
    def forward(self, x, t):
        emb = self.time_embed(t)
        return self.net(torch.cat([x, emb], dim=-1))

model = EpsNet()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 20000
BATCH = 128
losses = []
for epoch in range(EPOCHS):
    idx = np.random.choice(len(Xt), BATCH, replace=False)
    x_0 = Xt[idx]
    t = torch.randint(0, T, (BATCH,))
    eps = torch.randn_like(x_0)
    ab_t = alpha_bars[t].unsqueeze(-1)
    x_t = torch.sqrt(ab_t) * x_0 + torch.sqrt(1 - ab_t) * eps
    eps_pred = model(x_t, t)
    loss = ((eps - eps_pred) ** 2).sum(dim=-1).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if epoch % 2000 == 0:
        print(f'epoch {epoch}: loss {loss.item():.4f}')
print('final loss', losses[-1])

# Save weights
weights = {k: v.detach().numpy().tolist() for k, v in model.state_dict().items()}
weights['_metadata'] = {
    'T': T,
    'betas': betas.tolist(),
    'alpha_bars': alpha_bars.tolist(),
    'data_centers': centers.tolist(),
    'hidden_dim': HIDDEN,
    'time_dim': TIME_DIM,
    'epochs': EPOCHS,
    'schedule': 'linear-1e-4-2e-2',
}
out = Path('../src/lessons/ddpm/assets/ddpm-weights.json')
out.write_text(json.dumps(weights))
print('wrote', out)